In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion-turso-v3/notebooks")

# 02 · Medallion pipeline: Turso → Bronze → Silver → Gold
Everything runs inside Turso as SQL; the notebook only sends statements.

| Step | What | Turso tables |
|---|---|---|
| 1 | **Stage**: incremental copy from the source tables into Bronze (text + lineage) | `bronze_*` |
| 2 | **Clean**: cast, standardise, DQ rules (incl. de-duplication) → quarantine, latest per key, guarded upsert | `silver_*` |
| 3 | **Join**: each fact ⟕ its other Silver tables and conformed dimensions, surrogate keys, Unknown member | `wrk_fact_*` (work tables) |
| 4 | **Merge** into the Gold dimensions and facts (`fact_sales`, `fact_returns`) | `gold_*` |
| 5 | **Delta** (optional): only Silver rows changed since the last Gold load | `ctl_watermark` |

Run it after each source batch (notebook 01). `LOAD_MODE=full` rebuilds Gold from all of Silver.

Everything about the tables (columns, types, keys, dimensions, facts) comes from `domains/sales.yml`; the rules from `rules/dq_rules.yml`.

In [2]:
%run -i ./00_config.ipynb

rule library: 27 checks for 9 entities


In [3]:
SALES = load_domain("sales")                                  # entities, dimensions and facts from domains/sales.yml
open_warehouse()
new_batch()
print(f"run batch {BATCH_ID} | gold load mode: {LOAD_MODE} | warehouse: Turso")

domain 'sales': 4 entities, 2 dimensions, 2 facts
run batch 3 | gold load mode: incremental | warehouse: Turso


## 1 · Stage (Bronze)

In [4]:
for e in SALES["entities"]: stage(e)

  customers          extracted     1 rows (since 2026-10-30 14:13:55),     0 new in bronze
  products           extracted     1 rows (since 2026-10-31 18:47:42),     0 new in bronze
  sales              extracted     1 rows (since 2026-10-31 20:05:30),     0 new in bronze
  returns            extracted     1 rows (since 2026-10-28 14:00:00),     0 new in bronze


## 2 · Clean (Silver)
Includes the de-duplication rule `duplicate_order_line` from `dq_rules.yml`: of several sales lines with the same order, customer, product and order time, the first submitted is kept and the later copies are quarantined.

In [5]:
for e in SALES["entities"]: clean_to_silver(e)

  customers          nothing new in bronze
  products           nothing new in bronze
  sales              nothing new in bronze
  returns            nothing new in bronze


## 3 · Join
Dimensions first (so keys exist), then **each fact is joined on its own**: `build_fact(fact)` works the same for every fact in the YAML.

In [6]:
for d in SALES["dims"]: load_dim(d)
for f in SALES["facts"]:
    print(f"  {f:14s} join {build_fact(f):5d} rows ready (delta + any rows waiting on a late dimension)")

  dim_customer         0 inserted/updated
  dim_product          0 inserted/updated
  fact_sales     join    19 rows ready (delta + any rows waiting on a late dimension)
  fact_returns   join     4 rows ready (delta + any rows waiting on a late dimension)


## 4 · Merge into Gold
`merge_fact(fact)` per fact; add a fact to `domains/sales.yml` and it is picked up here with no code change.

In [7]:
for f in SALES["facts"]:
    print(f"  {f:14s} {merge_fact(f):5d} inserted/updated")

  fact_sales         0 inserted/updated
  fact_returns       0 inserted/updated


## 5 · Delta, watermarks & checks
Run the pipeline again with no new source data and every step reports "nothing new".

In [8]:
display(con.df("SELECT name, value, updated_ts FROM ctl_watermark WHERE name NOT LIKE '%customer_unified%' ORDER BY name"))
display(con.df("""
  SELECT 'silver' AS layer, 'sales' AS tbl, count(*) AS n FROM silver_sales UNION ALL
  SELECT 'silver', 'returns', count(*) FROM silver_returns UNION ALL
  SELECT 'silver', 'silver_quarantine (all entities)', count(*) FROM silver_quarantine UNION ALL
  SELECT 'gold', 'fact_sales', count(*) FROM gold_fact_sales UNION ALL
  SELECT 'gold', 'fact_sales with unknown dim key', count(*) FROM gold_fact_sales WHERE customer_sk = -1 OR product_sk = -1 UNION ALL
  SELECT 'gold', 'fact_returns', count(*) FROM gold_fact_returns UNION ALL
  SELECT 'gold', 'fact_returns with unknown dim key', count(*) FROM gold_fact_returns WHERE customer_sk = -1 OR product_sk = -1
"""))

,name,value,updated_ts
0,extract.billing_customers,2026-10-31 20:29:53,2026-11-19 02:43:45
1,extract.crm_accounts,2026-10-30 14:13:55,2026-11-19 02:43:45
2,extract.customers,2026-10-30 14:13:55,2026-11-19 03:11:44
3,extract.marketing_leads,2026-10-28 20:20:37,2026-11-19 02:43:46
4,extract.products,2026-10-31 18:47:42,2026-11-19 03:11:45
5,extract.returns,2026-10-28 14:00:00,2026-11-19 03:11:45
6,extract.sales,2026-10-31 20:05:30,2026-11-19 03:11:45
7,extract.support_contacts,2026-10-31 21:16:00,2026-11-19 02:43:45
8,extract.web_visitors,2026-10-31 18:55:13,2026-11-19 02:43:45
9,gold.dim_customer,1,2026-11-19 03:11:46


,layer,tbl,n
0,silver,sales,300
1,silver,returns,27
2,silver,silver_quarantine (all entities),23
3,gold,fact_sales,300
4,gold,fact_sales with unknown dim key,19
5,gold,fact_returns,27
6,gold,fact_returns with unknown dim key,4


In [9]:
display(con.df("SELECT entity, batch_id, _errors, payload FROM silver_quarantine ORDER BY batch_id, entity LIMIT 10"))
display(con.df("""
  SELECT f.order_date, c.name, c.segment, p.product_name, f.quantity, f.amount
  FROM gold_fact_sales f JOIN gold_dim_customer c USING (customer_sk) JOIN gold_dim_product p USING (product_sk)
  ORDER BY f.order_ts DESC LIMIT 10"""))
display(con.df("""
  SELECT r.return_date, c.name, p.product_name, r.reason, r.returned_quantity, r.refund_amount, r.days_to_return
  FROM gold_fact_returns r JOIN gold_dim_customer c USING (customer_sk) JOIN gold_dim_product p USING (product_sk)
  ORDER BY r.return_ts DESC LIMIT 10"""))

,entity,batch_id,_errors,payload
0,products,1,list_price_non_negative,"{""product_id"":""19"",""product_name"":""Webcam"",""ca..."
1,returns,1,returned_quantity_positive,"{""return_id"":""20027"",""order_line_id"":""10001"",""..."
2,returns,1,refund_non_negative,"{""return_id"":""20028"",""order_line_id"":""10005"",""..."
3,returns,1,return_ts_is_not_null,"{""return_id"":""20029"",""order_line_id"":""10006"",""..."
4,sales,1,quantity_positive,"{""order_line_id"":""10008"",""order_id"":""5003"",""cu..."
5,sales,1,duplicate_order_line,"{""order_line_id"":""20001"",""order_id"":""5003"",""cu..."
6,sales,1,duplicate_order_line,"{""order_line_id"":""20002"",""order_id"":""5003"",""cu..."
7,sales,1,duplicate_order_line,"{""order_line_id"":""20003"",""order_id"":""5006"",""cu..."
8,sales,1,duplicate_order_line,"{""order_line_id"":""20004"",""order_id"":""5012"",""cu..."
9,sales,1,quantity_positive,"{""order_line_id"":""10042"",""order_id"":""5016"",""cu..."


,order_date,name,segment,product_name,quantity,amount
0,2026-10-31,Jamal Silva,smb,Cable Pack,5,37.50
1,2026-10-31,Hector Brown,smb,Pro Kit,3,161.97
2,2026-10-31,Hector Brown,smb,Starter Kit,1,21.99
3,2026-10-31,Uma Garcia,retail,Cloud Seat,1,12.00
4,2026-10-31,Uma Nguyen,retail,Docking Station,3,348.30
5,2026-10-31,Bob Lee,enterprise,Analytics Add-on,3,161.70
6,2026-10-31,Hector Brown,smb,Backup Plan,4,60.00
7,2026-10-31,Uma Nguyen,retail,Gizmo,4,18.00
8,2026-10-31,Uma Garcia,retail,Widget,2,19.98
9,2026-10-31,Uma Nguyen,retail,Backup Plan,3,45.00


,return_date,name,product_name,reason,returned_quantity,refund_amount,days_to_return
0,2026-10-28,Quinn Lee,Monitor Stand,buyer remorse,1,20.00,37.0
1,2026-10-27,Vik Jones,Install Service,late_delivery,1,150.00,19.0
2,2026-10-20,None,None,damaged,1,25.00,NaN
3,2026-10-19,Sam Brown,Cloud Seat,wrong_item,1,10.80,10.0
4,2026-10-18,None,None,wrong_item,1,10.00,NaN
5,2026-10-18,Frank Meyer,Backup Plan,damaged,1,15.00,5.0
6,2026-10-17,Cara Novak,Docking Station,changed_mind,4,567.60,13.0
7,2026-10-17,Zoe Khan,Keyboard,late_delivery,5,225.00,4.0
8,2026-10-16,Frank Meyer,Cable Pack,not_as_described,1,7.50,3.0
9,2026-10-15,Pavel Cohen,Cable Pack,wrong_item,1,8.25,18.0


## 6 · De-duplication: the duplicates that were removed
Each row below was rejected by the `duplicate_order_line` rule; `kept_line` is the first-submitted line that survived with the same order, customer, product and time.

In [10]:
display(con.df("""
  SELECT q.batch_id, json_extract(q.payload, '$.order_line_id') AS rejected_line, d.order_line_id AS kept_line,
         json_extract(q.payload, '$.order_id') AS order_id, json_extract(q.payload, '$.customer_id') AS customer_id,
         json_extract(q.payload, '$.product_id') AS product_id, q._errors
  FROM silver_quarantine q
  JOIN silver_sales d ON d.order_id = CAST(json_extract(q.payload, '$.order_id') AS INTEGER)
                     AND d.customer_id = CAST(json_extract(q.payload, '$.customer_id') AS INTEGER)
                     AND d.product_id = CAST(json_extract(q.payload, '$.product_id') AS INTEGER)
                     AND d.order_ts = strftime('%Y-%m-%d %H:%M:%S', json_extract(q.payload, '$.order_ts'))
  WHERE q.entity = 'sales' AND q._errors LIKE '%duplicate_order_line%'
  ORDER BY q.batch_id, kept_line, rejected_line"""))

,batch_id,rejected_line,kept_line,order_id,customer_id,product_id,_errors
0,1,20001,10005,5003,42,15,duplicate_order_line
1,1,20002,10005,5003,42,15,duplicate_order_line
2,1,20003,10015,5006,39,18,duplicate_order_line
3,1,20004,10033,5012,49,5,duplicate_order_line
4,1,20005,10051,5020,43,11,duplicate_order_line
5,1,20006,10072,5029,59,17,duplicate_order_line


## 7 · Data-quality report (rules from `dq_rules.yml`)
Failed vs checked rows per rule, summed over all runs. `error` rules quarantine the row (see `_errors` above); `warn` rules only count.

In [11]:
display(con.df("""SELECT entity, rule, criticality, function, sum(failed_rows) AS failed_rows, sum(checked_rows) AS checked_rows
                  FROM silver_dq_results GROUP BY entity, rule, criticality, function ORDER BY entity, criticality, rule"""))

,entity,rule,criticality,function,failed_rows,checked_rows
0,billing_customers,billing_id_is_not_null,error,is_not_null,0,47
1,billing_customers,email_is_not_null,error,is_not_null,0,47
2,billing_customers,mrr_is_not_null,error,is_not_null,0,47
3,billing_customers,mrr_non_negative,error,is_not_less_than,1,47
4,billing_customers,updated_at_is_not_null,error,is_not_null,0,47
5,billing_customers,billing_plan_known,warn,is_in_list,0,47
6,billing_customers,billing_status_known,warn,is_in_list,0,47
7,crm_accounts,account_id_is_not_null,error,is_not_null,0,60
8,crm_accounts,email_is_not_null,error,is_not_null,4,60
9,crm_accounts,updated_at_is_not_null,error,is_not_null,0,60


### Optional · Prove the YAML runs unchanged on native DQX (Spark)
Runs the same `dq_rules.yml` with `DQEngineCore` on a local Spark session and compares the error / warning row counts with the SQL executor (for `is_unique` it compares the rows DQX puts in duplicate groups with the SQL's groups). Needs Java; if Spark cannot start the cell just prints why and is skipped.

In [12]:
for e in SALES["entities"]: native_dqx_check(e)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 03:11:49 WARN Utils: Your hostname, codespaces-f42fe3, resolves to a loopback address: 127.0.0.1; using 10.0.2.156 instead (on interface eth0)
26/11/19 03:11:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 03:11:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


  customers          SQL errors/warnings = 0/0 | native DQX = 0/0 -> MATCH
  products           SQL errors/warnings = 1/0 | native DQX = 1/0 -> MATCH
  sales              SQL errors/warnings = 4/0 | native DQX = 4/0 | rows in duplicate groups SQL/DQX = 11/11 -> MATCH
  returns            SQL errors/warnings = 3/1 | native DQX = 3/1 -> MATCH


## Scaling notes
* **Where the work happens:** every step is a set-based SQL statement inside Turso; no rows travel to the notebook, so notebook memory does not grow with volume.
* **Source volume:** extraction is keyset-style (`updated_at` watermark, indexed in the source schema). Turso is built for operational-size data; when the warehouse outgrows it (hundreds of millions of rows, heavy analytics), point the same YAML-driven design at a columnar engine (MotherDuck, Databricks).
* **Watermark caveat:** the extract assumes `updated_at` is set on every insert and update. Deletes are not seen; add an `is_deleted` flag or a tombstone table if you need them.
* **De-duplication caveat:** the first submitted record wins. Duplicates are found among the new rows of a run and against what is already in Silver; a record already in Silver is never removed because a later copy arrives.
* **Orchestration:** the steps are plain functions; call them from GitHub Actions on a schedule (`jupyter nbconvert --execute`) or from any scheduler.
* **Porting back to Databricks:** the metadata, step order and watermark logic carry over unchanged; the helpers become Auto Loader, Delta `MERGE` and Change Data Feed.

In [13]:
release_all()          # closes the Turso connection (the next notebook opens its own)